# Protocol Comparison — Shuffled vs. Legacy vs. Grouped CV

Built for the response letter: a direct, side-by-side comparison of the four evaluation
protocols on the *same* two models (Extra Trees on FE features, CatBoost on Base
features), with **no hyperparameter tuning** — fixed, reasonable defaults — so that
differences between columns are due to the CV protocol alone, not to search variance.

| Protocol | Splitter | Leaks within-session rows? |
|---|---|---|
| `shuffled` | `StratifiedKFold(5, shuffle=True)` | yes — explicit leak baseline |
| `legacy unshuffled` | `StratifiedKFold(5, shuffle=False)` | mostly not, by accident (old Table 1 protocol) |
| `primary` | `LeaveOneGroupOut` (task-specific group) | no |
| `LOSO` | `LeaveOneGroupOut(groups=session)` / `GroupKFold(10, groups=session)` | no |

See `../src/eval_protocol.py` and the main README for the full protocol definitions.

## 1. Imports & Dataset

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_predict

sys.path.insert(0, str(Path.cwd().parent / "src"))
import eval_protocol as ep

REPO_ROOT = ep._find_repo_root()
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

PROTOCOL_LABELS = {
    "shuffled": "Shuffled 5-fold",
    "legacy_unshuffled": "Legacy unshuffled 5-fold",
    "primary": "Primary (grouped LOGO)",
    "loso": "LOSO (session)",
}

## 2. Fixed-Hyperparameter Models

Untuned, fixed-hyperparameter models — the same `ExtraTreesClassifier(n_estimators=300,
max_features="sqrt")` used for the acceptance-check sanity values, `CatBoostClassifier`
at reasonable defaults, and a one-node `DecisionTreeClassifier(max_depth=1)` baseline for
Attack Detection (a single RSSI threshold on `mean_rssi`).

In [2]:
def make_et():
    return ExtraTreesClassifier(n_estimators=300, max_features="sqrt", random_state=ep.RANDOM_STATE)

def make_catboost():
    # eval_protocol._CatBoostFlat, not the raw CatBoostClassifier:
    # CatBoostClassifier.predict() always returns shape (n, 1), which silently
    # breaks any plain `pred == y` comparison via numpy broadcasting (it degrades
    # to a spurious ~chance-level "accuracy"). _CatBoostFlat.predict() ravels
    # it to (n,), consistent with every other sklearn classifier's predict().
    return ep._CatBoostFlat(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=ep.RANDOM_STATE, verbose=False, allow_writing_files=False,
        thread_count=1,
    )

def make_baseline():
    return DecisionTreeClassifier(max_depth=1, random_state=ep.RANDOM_STATE)

MODEL_RUNS = {
    "attack": [("Extra Trees (FE)", make_et, ep.FEATURES_FE),
               ("CatBoost (Base)", make_catboost, ep.FEATURES_BASE),
               ("Threshold baseline (mean_rssi)", make_baseline, ["mean_rssi"])],
    "point": [("Extra Trees (FE)", make_et, ep.FEATURES_FE),
              ("CatBoost (Base)", make_catboost, ep.FEATURES_BASE)],
    "antenna": [("Extra Trees (FE)", make_et, ep.FEATURES_FE),
                ("CatBoost (Base)", make_catboost, ep.FEATURES_BASE)],
}

## 3. Run All Protocols x Models x Tasks, at 5s and 10s Resampling

In [3]:
def run_protocol_comparison(freq):
    raw = ep.load_raw()
    dataset, info = ep.build_dataset(raw, freq=freq)
    dataset = ep.add_fe(dataset)

    rows = []
    for task in ep.TASKS_ORDER:
        spec = ep.TASKS[task]
        sub = dataset[spec["rows"](dataset)].reset_index(drop=True)
        y = sub[spec["y_col"]]

        for model_label, make_model, feat_cols in MODEL_RUNS[task]:
            X = sub[feat_cols]
            for proto in ["shuffled", "legacy_unshuffled", "primary", "loso"]:
                cv, groups = ep.get_outer_cv_and_groups(proto, task, sub)
                if groups is not None:
                    pred = cross_val_predict(make_model(), X, y, cv=cv, groups=groups, n_jobs=-1)
                else:
                    pred = cross_val_predict(make_model(), X, y, cv=cv, n_jobs=-1)
                acc = float((pred == y.to_numpy()).mean())
                from sklearn.metrics import f1_score
                f1 = float(f1_score(y, pred, average="macro"))
                rows.append({
                    "Freq": freq, "Task": spec["label"], "Model": model_label,
                    "Protocol": PROTOCOL_LABELS[proto], "Accuracy": round(acc, 4), "Macro F1": round(f1, 4),
                })
    return pd.DataFrame(rows)

comparison_5s = run_protocol_comparison("5s")
comparison_5s

,Freq,Task,Model,Protocol,Accuracy,Macro F1
0,5s,Attack Detection,Extra Trees (FE),Shuffled 5-fold,0.9855,0.9815
1,5s,Attack Detection,Extra Trees (FE),Legacy unshuffled 5-fold,0.9883,0.9851
2,5s,Attack Detection,Extra Trees (FE),Primary (grouped LOGO),0.9878,0.9845
3,5s,Attack Detection,Extra Trees (FE),LOSO (session),0.9855,0.9815
4,5s,Attack Detection,CatBoost (Base),Shuffled 5-fold,0.9873,0.9839
5,5s,Attack Detection,CatBoost (Base),Legacy unshuffled 5-fold,0.9878,0.9845
6,5s,Attack Detection,CatBoost (Base),Primary (grouped LOGO),0.9878,0.9845
7,5s,Attack Detection,CatBoost (Base),LOSO (session),0.9873,0.9838
8,5s,Attack Detection,Threshold baseline (mean_rssi),Shuffled 5-fold,0.9859,0.9821
9,5s,Attack Detection,Threshold baseline (mean_rssi),Legacy unshuffled 5-fold,0.9883,0.9851


In [4]:
comparison_10s = run_protocol_comparison("10s")
comparison_10s

,Freq,Task,Model,Protocol,Accuracy,Macro F1
0,10s,Attack Detection,Extra Trees (FE),Shuffled 5-fold,0.9747,0.9678
1,10s,Attack Detection,Extra Trees (FE),Legacy unshuffled 5-fold,0.9728,0.9655
2,10s,Attack Detection,Extra Trees (FE),Primary (grouped LOGO),0.9756,0.9691
3,10s,Attack Detection,Extra Trees (FE),LOSO (session),0.9719,0.9642
4,10s,Attack Detection,CatBoost (Base),Shuffled 5-fold,0.9737,0.9666
5,10s,Attack Detection,CatBoost (Base),Legacy unshuffled 5-fold,0.9719,0.9640
6,10s,Attack Detection,CatBoost (Base),Primary (grouped LOGO),0.9737,0.9665
7,10s,Attack Detection,CatBoost (Base),LOSO (session),0.9681,0.9592
8,10s,Attack Detection,Threshold baseline (mean_rssi),Shuffled 5-fold,0.9747,0.9677
9,10s,Attack Detection,Threshold baseline (mean_rssi),Legacy unshuffled 5-fold,0.9756,0.9689


## 4. Combined Table

In [5]:
protocol_comparison_df = pd.concat([comparison_5s, comparison_10s], ignore_index=True)
protocol_comparison_df.to_csv(RESULTS_DIR / "protocol_comparison.csv", index=False)
protocol_comparison_df

,Freq,Task,Model,Protocol,Accuracy,Macro F1
0,5s,Attack Detection,Extra Trees (FE),Shuffled 5-fold,0.9855,0.9815
1,5s,Attack Detection,Extra Trees (FE),Legacy unshuffled 5-fold,0.9883,0.9851
2,5s,Attack Detection,Extra Trees (FE),Primary (grouped LOGO),0.9878,0.9845
3,5s,Attack Detection,Extra Trees (FE),LOSO (session),0.9855,0.9815
4,5s,Attack Detection,CatBoost (Base),Shuffled 5-fold,0.9873,0.9839
5,5s,Attack Detection,CatBoost (Base),Legacy unshuffled 5-fold,0.9878,0.9845
6,5s,Attack Detection,CatBoost (Base),Primary (grouped LOGO),0.9878,0.9845
7,5s,Attack Detection,CatBoost (Base),LOSO (session),0.9873,0.9838
8,5s,Attack Detection,Threshold baseline (mean_rssi),Shuffled 5-fold,0.9859,0.9821
9,5s,Attack Detection,Threshold baseline (mean_rssi),Legacy unshuffled 5-fold,0.9883,0.9851


### 4.1 Pivoted view (5s grid, accuracy only)

Rows = task/model, columns = protocol — this is the table format used in the response
letter.

In [6]:
pivot = comparison_5s.pivot(index=["Task", "Model"], columns="Protocol", values="Accuracy")
pivot = pivot[[PROTOCOL_LABELS[p] for p in ["shuffled", "legacy_unshuffled", "primary", "loso"]]]
pivot

Protocol                                           Shuffled 5-fold  \
Task               Model                                             
Antenna Prediction CatBoost (Base)                          0.7881   
                   Extra Trees (FE)                         0.8231   
Attack Detection   CatBoost (Base)                          0.9873   
                   Extra Trees (FE)                         0.9855   
                   Threshold baseline (mean_rssi)           0.9859   
Point Prediction   CatBoost (Base)                          0.8914   
                   Extra Trees (FE)                         0.8967   

Protocol                                           Legacy unshuffled 5-fold  \
Task               Model                                                      
Antenna Prediction CatBoost (Base)                                   0.4343   
                   Extra Trees (FE)                                  0.4816   
Attack Detection   CatBoost (Base)                                   0.9878   
                   Extra Trees (FE)                                  0.9883   
                   Threshold baseline (mean_rssi)                    0.9883   
Point Prediction   CatBoost (Base)                                   0.8074   
                   Extra Trees (FE)                                  0.8161   

Protocol                                           Primary (grouped LOGO)  \
Task               Model                                                    
Antenna Prediction CatBoost (Base)                                 0.3187   
                   Extra Trees (FE)                                0.3363   
Attack Detection   CatBoost (Base)                                 0.9878   
                   Extra Trees (FE)                                0.9878   
                   Threshold baseline (mean_rssi)                  0.9883   
Point Prediction   CatBoost (Base)                                 0.7268   
                   Extra Trees (FE)                                0.7356   

Protocol                                           LOSO (session)  
Task               Model                                           
Antenna Prediction CatBoost (Base)                         0.1121  
                   Extra Trees (FE)                        0.1156  
Attack Detection   CatBoost (Base)                         0.9873  
                   Extra Trees (FE)                        0.9855  
                   Threshold baseline (mean_rssi)          0.9883  
Point Prediction   CatBoost (Base)                         0.6830  
                   Extra Trees (FE)                        0.6900